In [2]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print(os.environ["HADOOP_HOME"])
print(os.environ["PATH"].split(";")[:5])

C:\hadoop
['C:\\hadoop\\bin', 'C:\\hadoop\\bin', 'c:\\Users\\m24810\\Projects\\beverage-sales\\.venv\\Scripts', 'C:\\Users\\m24810\\Projects\\beverage-sales\\.venv\\Scripts', 'C:\\Users\\m24810\\.local\\bin']


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, StringType, DecimalType, IntegerType, DateType)
from pyspark.sql import functions as F
from pyspark.sql.functions import round

print("1. Antes de criar SparkSession")
spark = SparkSession.builder.appName("beverage-sales").getOrCreate()
print("2. SparkSession criada")

print("3. Antes de ler CSV")
schema = StructType([
    StructField("Order_ID", StringType(), True),
    StructField("Customer_ID", StringType(), True),
    StructField("Customer_Type", StringType(), True),
    StructField("Product", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Unit_Price", DecimalType(10, 2), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Discount", DecimalType(5, 2), True),
    StructField("Total_Price", DecimalType(12, 2), True),
    StructField("Region", StringType(), True),
    StructField("Order_Date", DateType(), True)
])

df_raw = spark.read \
    .format("csv") \
    .option("header", True) \
    .option("delimiter", ",") \
    .option("mode", "FAILFAST") \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(schema) \
    .load("C:/Users/m24810/Projects/beverage-sales/data/raw/synthetic_beverage_sales_data.csv")

#.load("C:/Users/m24810/Projects/beverage-sales/data/raw/synthetic_beverage_sales_data.csv")
#.load("/home/rgerc/Projects/beverage-sales/data/raw/synthetic_beverage_sales_data.csv")
print("4. CSV lido")

print("5. Antes de count")
print(df_raw.count())
print("6. Count terminado")


#------------------------Data Cleaning------------------------#

print("Verify if there are any null values in the dataset:")
df_raw.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
]).show()

print("Check for duplicate Order_IDs (it could make sense to have multiple entries for the same order):")
df_raw.groupBy("Order_ID").count().orderBy(F.desc("count")).show()

print("Check if Total_Price is correctly calculated:")
df_check = df_raw.withColumn(
    "Expected_Total",
    F.round(
        F.col("Unit_Price") * F.col("Quantity") * (1 - F.col("Discount")),
        2
    )
)

print("Show rows where Total_Price does not match the expected calculation:")
df_check.filter(
    F.abs(F.col("Expected_Total") - F.col("Total_Price")) > 0.01
).show()

print("Check for negative or zero values in Quantity and Unit_Price, and invalid Discount values:")
df_raw.filter(F.col("Quantity") <= 0).show()

df_raw.filter(F.col("Unit_Price") <= 0).show()

df_raw.filter((F.col("Discount") < 0) | (F.col("Discount") > 1)).show()

print("Some columns need to be derived for easier analysis:")
df_clean = df_raw \
    .withColumn("Year", F.year("Order_Date")) \
    .withColumn("Month", F.month("Order_Date")) \
    .withColumn("Gross_Sales", F.col("Unit_Price") * F.col("Quantity")) \
    .withColumn("Discount_Value", F.col("Gross_Sales") * F.col("Discount"))

df_clean.show(5)


1. Antes de criar SparkSession
2. SparkSession criada
3. Antes de ler CSV
4. CSV lido
5. Antes de count
8999910
6. Count terminado
Verify if there are any null values in the dataset:
+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+
|Order_ID|Customer_ID|Customer_Type|Product|Category|Unit_Price|Quantity|Discount|Total_Price|Region|Order_Date|
+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+
|       0|          0|            0|      0|       0|         0|       0|       0|          0|     0|         0|
+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+

Check for duplicate Order_IDs (it could make sense to have multiple entries for the same order):
+--------+-----+
|Order_ID|count|
+--------+-----+
|ORD10894|    5|
| ORD9718|    5|
| ORD9669|    5|
| ORD1461|    5|
|ORD11013|    5|
| ORD4186|    5|
|

In [6]:
df_clean.select("Category").distinct().show()

+-------------------+
|           Category|
+-------------------+
|              Water|
|Alcoholic Beverages|
|        Soft Drinks|
|             Juices|
+-------------------+



In [7]:
df_clean.select("Product").distinct().count()

47

In [8]:
df_clean.select("Customer_Type").distinct().show()

+-------------+
|Customer_Type|
+-------------+
|          B2C|
|          B2B|
+-------------+



In [9]:
df_clean.select("Region").distinct().show()

+--------------------+
|              Region|
+--------------------+
|      Sachsen-Anhalt|
|       Niedersachsen|
|         Brandenburg|
|              Berlin|
|              Bayern|
|             Sachsen|
|             Hamburg|
|              Bremen|
| Nordrhein-Westfalen|
|           Thüringen|
|              Hessen|
|   Baden-Württemberg|
|Mecklenburg-Vorpo...|
|            Saarland|
|  Schleswig-Holstein|
|     Rheinland-Pfalz|
+--------------------+



In [13]:
#------------------------First Analysis------------------------#
print("Sales by Category:")
df_clean.groupBy("Category") \
    .agg(F.sum("Total_Price").alias("total_sales")) \
    .orderBy(F.desc("total_sales")) \
    .show()

print("Sales by Region:")
df_clean.groupBy("Region") \
    .agg(F.sum("Total_Price").alias("sales")) \
    .orderBy(F.desc("sales")) \
    .show()

print("Monthly Sales:")
df_clean.groupBy("year", "month") \
    .agg(F.sum("Total_Price").alias("sales")) \
    .orderBy("year", "month") \
    .show()

Sales by Category:
+-------------------+------------+
|           Category| total_sales|
+-------------------+------------+
|Alcoholic Beverages|911797919.72|
|             Juices|133167848.64|
|        Soft Drinks| 82802542.53|
|              Water| 48912852.46|
+-------------------+------------+

Sales by Region:
+--------------------+-----------+
|              Region|      sales|
+--------------------+-----------+
|             Hamburg|82470771.98|
|              Hessen|78400110.19|
|            Saarland|78390587.15|
|     Rheinland-Pfalz|75838676.99|
|Mecklenburg-Vorpo...|75517247.16|
|           Thüringen|75324865.47|
|              Berlin|74567927.53|
|              Bayern|72825399.18|
|       Niedersachsen|71959050.52|
|             Sachsen|71946991.38|
|   Baden-Württemberg|71594838.90|
| Nordrhein-Westfalen|71393803.70|
|         Brandenburg|71349031.11|
|  Schleswig-Holstein|70317829.38|
|      Sachsen-Anhalt|69765543.62|
|              Bremen|65018489.09|
+-----------------

In [14]:
# Data Profiling: Basic Information
print("Number of rows:", df_clean.count())
print("Number of columns:", len(df_clean.columns))
print("Columns:", df_clean.columns)
print("\nSchema:")
df_clean.printSchema()

Number of rows: 8999910
Number of columns: 15
Columns: ['Order_ID', 'Customer_ID', 'Customer_Type', 'Product', 'Category', 'Unit_Price', 'Quantity', 'Discount', 'Total_Price', 'Region', 'Order_Date', 'Year', 'Month', 'Gross_Sales', 'Discount_Value']

Schema:
root
 |-- Order_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Type: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Unit_Price: decimal(10,2) (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: decimal(5,2) (nullable = true)
 |-- Total_Price: decimal(12,2) (nullable = true)
 |-- Region: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Gross_Sales: decimal(21,2) (nullable = true)
 |-- Discount_Value: decimal(27,4) (nullable = true)



In [16]:
# Data Profiling: Summary Statistics
print("Summary Statistics:")
df_clean.describe("Unit_Price", "Quantity", "Discount", "Total_Price", "Gross_Sales", "Discount_Value").show()

Summary Statistics:
+-------+------------------+------------------+-------------------+------------------+-----------------+-----------------+
|summary|        Unit_Price|          Quantity|           Discount|       Total_Price|      Gross_Sales|   Discount_Value|
+-------+------------------+------------------+-------------------+------------------+-----------------+-----------------+
|  count|           8999910|           8999910|            8999910|           8999910|          8999910|          8999910|
|   mean|          5.818037| 23.13813371467048|           0.029729|        130.743659|       141.456051|      10.71237767|
| stddev|14.700500712480949|26.893207084620304|0.04479840724082398|509.69474263281745|566.3716609799259|61.12826606335841|
|    min|              0.32|                 1|               0.00|              0.30|             0.32|           0.0000|
|    max|            169.53|               100|               0.15|          14295.30|         16818.00|        2522.70

In [17]:
# Data Profiling: Missing Values
from pyspark.sql.functions import col, sum as spark_sum

print("Missing Values per Column:")
df_clean.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_clean.columns]).show()

Missing Values per Column:
+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+----+-----+-----------+--------------+
|Order_ID|Customer_ID|Customer_Type|Product|Category|Unit_Price|Quantity|Discount|Total_Price|Region|Order_Date|Year|Month|Gross_Sales|Discount_Value|
+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+----+-----+-----------+--------------+
|       0|          0|            0|      0|       0|         0|       0|       0|          0|     0|         0|   0|    0|          0|             0|
+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+----+-----+-----------+--------------+



In [18]:
# Data Profiling: Unique Values
print("Unique Values per Column:")
for col_name in df_clean.columns:
    unique_count = df_clean.select(col_name).distinct().count()
    print(f"{col_name}: {unique_count} unique values")

Unique Values per Column:
Order_ID: 3000000 unique values
Customer_ID: 10000 unique values
Customer_Type: 2 unique values
Product: 47 unique values
Category: 4 unique values
Unit_Price: 12778 unique values
Quantity: 100 unique values
Discount: 4 unique values
Total_Price: 203842 unique values
Region: 16 unique values
Order_Date: 1094 unique values
Year: 3 unique values
Month: 12 unique values
Gross_Sales: 174157 unique values
Discount_Value: 203032 unique values


In [19]:
# Data Profiling: Distributions
# For categorical columns
print("Top 10 Categories:")
df_clean.groupBy("Category").count().orderBy("count", ascending=False).show(10)

print("Top 10 Regions:")
df_clean.groupBy("Region").count().orderBy("count", ascending=False).show(10)

# For numerical columns, quantiles
print("Quantiles for Total_Price:")
quantiles = df_clean.approxQuantile("Total_Price", [0.25, 0.5, 0.75], 0.05)
print(f"25%: {quantiles[0]}, 50%: {quantiles[1]}, 75%: {quantiles[2]}")

print("Quantiles for Quantity:")
quantiles_qty = df_clean.approxQuantile("Quantity", [0.25, 0.5, 0.75], 0.05)
print(f"25%: {quantiles_qty[0]}, 50%: {quantiles_qty[1]}, 75%: {quantiles_qty[2]}")

Top 10 Categories:
+-------------------+-------+
|           Category|  count|
+-------------------+-------+
|Alcoholic Beverages|2251625|
|              Water|2250217|
|             Juices|2249937|
|        Soft Drinks|2248131|
+-------------------+-------+

Top 10 Regions:
+-------------------+------+
|             Region| count|
+-------------------+------+
|            Hamburg|604054|
|    Rheinland-Pfalz|577967|
|      Niedersachsen|577005|
|           Saarland|573596|
|            Sachsen|572827|
|Nordrhein-Westfalen|568797|
|             Bremen|568223|
| Schleswig-Holstein|565825|
|          Thüringen|562554|
|             Bayern|556069|
+-------------------+------+
only showing top 10 rows
Quantiles for Total_Price:
25%: 8.68, 50%: 20.52, 75%: 55.33
Quantiles for Quantity:
25%: 6.0, 50%: 11.0, 75%: 20.0


In [20]:
# Data Profiling: Correlations
print("Correlation between Quantity and Total_Price:")
corr = df_clean.stat.corr("Quantity", "Total_Price")
print(f"Correlation: {corr}")

# If there are other numerical columns, add more
# For example, if there's Unit_Price
try:
    corr_unit = df_clean.stat.corr("Unit_Price", "Total_Price")
    print(f"Correlation between Unit_Price and Total_Price: {corr_unit}")
except:
    print("Unit_Price not found or not numerical")

Correlation between Quantity and Total_Price:
Correlation: 0.3139230079357313
Correlation between Unit_Price and Total_Price: 0.6206489526938138


In [21]:
# Data Profiling: Outliers Detection
# Using IQR for Total_Price
quantiles = df_clean.approxQuantile("Total_Price", [0.25, 0.75], 0.05)
Q1 = quantiles[0]
Q3 = quantiles[1]
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"IQR Bounds for Total_Price: Lower={lower_bound}, Upper={upper_bound}")

outliers = df_clean.filter((col("Total_Price") < lower_bound) | (col("Total_Price") > upper_bound))
print(f"Number of outliers in Total_Price: {outliers.count()}")

# Show some outliers
outliers.select("Total_Price").show(5)

IQR Bounds for Total_Price: Lower=-61.294999999999995, Upper=125.30499999999999
Number of outliers in Total_Price: 1451471
+-----------+
|Total_Price|
+-----------+
|     126.36|
|     170.98|
|    1219.31|
|     246.20|
|     141.47|
+-----------+
only showing top 5 rows
